# LoRA Bench — Day 2: LoRA/QLoRA Fine-Tuning

Fine-tunes `Qwen/Qwen2.5-Coder-1.5B-Instruct` with QLoRA on the CVE
fix-diff dataset prepared by this repo's Day 1 data-prep pipeline, runs a
small LoRA rank sweep, then does the full fine-tune with the winning
config and saves the adapter. It defaults to a two-step **T4 smoke
test**; set `SMOKE_TEST = False` only after that artifact has been reviewed.

Designed for the **free Colab T4 tier** — no paid tier, no external paid
API. See the repo's `README.md`/`ADR.md` for the full project context;
this notebook is one stage of `data prep -> fine-tune -> quantize ->
benchmark -> report` (Day 3/4 add the rest, in this same notebook).

**Rough total runtime estimate on a T4** (data prep + sweep + full
fine-tune + a quick qualitative check): well under an hour. This is an
estimate, not a measurement — this notebook can't be run outside Colab to
verify it, so treat the first run as the actual measurement.


## Before you start

1. **Runtime > Change runtime type > T4 GPU**, then re-run from the top.
2. **Optional**: add an `HF_TOKEN` secret (key icon, left sidebar) — a
   free, read-scope Hugging Face token. Neither the dataset
   (`hitoshura25/cvefixes`) nor the base model is gated, so this only
   raises Hub rate limits; the notebook runs fine without it.
3. **The clone cell below needs this repo to be reachable at the URL you
   set in `REPO_URL`.** If it's private, either make it public or clone
   via a token (`https://<token>@github.com/...` — paste the token
   through Colab Secrets, never hardcode it in a cell).
4. Run cells top to bottom. Nothing here needs manual babysitting once
   started, beyond watching for the sweep/training progress bars.


In [ ]:
import subprocess
import time

import torch

RUN_STARTED_AT = time.time()
GPU_INFO = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True).stdout
print(GPU_INFO)

assert torch.cuda.is_available(), (
    "No GPU detected. In Colab: Runtime > Change runtime type > T4 GPU, "
    "then Runtime > Restart session, then re-run from the top."
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")

In [ ]:
import os

# Update if you forked/renamed the repo, or see "Before you start" above
# for how to clone a private repo.
SMOKE_TEST = True  # Keep True until the returned smoke artifact has been reviewed.
REPO_URL = "https://github.com/HarshTikone/LoRA-bench.git"
REPO_DIR = "/content/lora-bench"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

# Repo-side deps (datasets, huggingface_hub, python-dotenv, PyYAML) come
# from pyproject.toml via the editable install. GPU-side deps are listed
# separately in requirements-colab.txt and installed explicitly here --
# torch itself is deliberately NOT reinstalled, to avoid fighting Colab's
# preinstalled CUDA-matched build. jinja2 is listed explicitly even though
# it happens to already be present (pulled in transitively by Colab's
# preinstalled torch) -- tokenizer.apply_chat_template needs it directly,
# and this notebook shouldn't depend on that staying true by accident.
!pip install -q -e .
!pip install -q -U transformers peft bitsandbytes accelerate jinja2
print(f"SMOKE_TEST={SMOKE_TEST}")

In [ ]:
import os

from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face Hub.")
else:
    print(
        "No HF_TOKEN found (Colab Secrets panel, key icon in the left sidebar). "
        "Continuing without it -- the dataset and base model are both public, "
        "so this only affects Hub rate limits, not whether this notebook runs."
    )

## 1. Data prep

Runs the exact same, already-unit-tested pipeline from Day 1
(`src/lora_bench/data/cvefixes.py`) against the live, pinned dataset
revision (see `ADR.md`'s ADR-0002) -- nothing about this step is
Colab-specific, it's just running repo code that needs network access
this environment doesn't restrict.


In [ ]:
!python -m lora_bench.data.cvefixes --config configs/default.yaml --out-dir data/processed

In [ ]:
import json
import platform
import sys

with open("data/processed/manifest.json") as f:
    manifest = json.load(f)

RUN_METADATA = {
    "smoke_test": SMOKE_TEST,
    "python": sys.version,
    "platform": platform.platform(),
    "git_sha": subprocess.run(
        ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
    ).stdout.strip(),
    "gpu_info": GPU_INFO,
}
print(json.dumps(manifest, indent=2))
print(json.dumps(RUN_METADATA, indent=2))

## 2. Tokenizer & tokenized datasets

The tested package-level preprocessing renders the generation prompt and
full conversation separately, verifies the prompt is an exact token
prefix, and masks prompt labels with `-100`. Examples over `max_seq_len`
are dropped and counted—fixed-code responses are never truncated. Smoke
mode limits training to 64 retained examples and validation to 16.


In [ ]:
from transformers import AutoTokenizer

from lora_bench.config import load_config

cfg = load_config("configs/default.yaml")
BASE_MODEL = cfg.model.base_model
MAX_SEQ_LEN = cfg.model.max_seq_len

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"base model: {BASE_MODEL}")
print(f"max_seq_len: {MAX_SEQ_LEN}")
print(f"pad_token: {tokenizer.pad_token!r}")

In [ ]:
from collections import Counter

from datasets import Dataset

from lora_bench.data.cvefixes import read_jsonl
from lora_bench.training import IGNORE_INDEX, tokenize_training_example


def assert_disjoint_cves(*splits):
    id_sets = [{example.cve_id for example in split} for split in splits]
    for i, ids in enumerate(id_sets):
        for other in id_sets[i + 1 :]:
            assert ids.isdisjoint(other), "CVE leakage detected across data splits"


def prepare_split(examples, limit=None):
    records = []
    counts = Counter()
    eligible = 0
    for example in examples:
        outcome = tokenize_training_example(example, tokenizer, MAX_SEQ_LEN)
        if not outcome.kept:
            counts[outcome.drop_reason.value] += 1
            continue
        assert any(label != IGNORE_INDEX for label in outcome.record["labels"])
        eligible += 1
        if limit is None or len(records) < limit:
            records.append(outcome.record)
    assert records, "No examples survived exact tokenization checks"
    return Dataset.from_list(records), dict(counts), eligible


train_examples = read_jsonl("data/processed/train.jsonl")
val_examples_all = read_jsonl("data/processed/val.jsonl")
test_examples = read_jsonl("data/processed/test.jsonl")
assert_disjoint_cves(train_examples, val_examples_all, test_examples)
train_ds, train_token_drops, train_eligible = prepare_split(
    train_examples, 64 if SMOKE_TEST else None
)
val_ds, val_token_drops, val_eligible = prepare_split(val_examples_all, 16 if SMOKE_TEST else None)
PREPROCESSING_COUNTERS = {
    "train": {
        "source": len(train_examples),
        "eligible": train_eligible,
        "retained": len(train_ds),
        "dropped": train_token_drops,
    },
    "val": {
        "source": len(val_examples_all),
        "eligible": val_eligible,
        "retained": len(val_ds),
        "dropped": val_token_drops,
    },
}
assert not SMOKE_TEST or len(train_ds) <= 64
assert len(val_ds) > 0
with open("/content/preprocessing_counters.json", "w") as f:
    json.dump(PREPROCESSING_COUNTERS, f, indent=2)
print(json.dumps(PREPROCESSING_COUNTERS, indent=2))

## 3. Base model loading (4-bit QLoRA)

`load_base_model()` is a function, not a one-off cell, because the sweep
below needs a **fresh** quantized base model per candidate: reusing one
base model object across multiple `get_peft_model()` calls risks stacking
adapters instead of cleanly replacing one, which isn't worth the risk of
a subtle bug for the ~30-60s a full reload costs on a 1.5B model.


In [ ]:
import torch
from peft import prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# T4 is Turing-generation and doesn't support bf16 compute well; detect
# rather than hardcode, so this also works correctly on newer GPUs.
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"compute dtype: {COMPUTE_DTYPE}")


def load_base_model():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
    )
    return prepare_model_for_kbit_training(model)

## 4. LoRA hyperparameter sweep

The past-"just a demo" checklist requires the *final* rank/hyperparameters
be a defended decision from an actual sweep, not just a logged default
(see `src/lora_bench/config.py`'s `LoRAConfig` docstring). Three
candidates spanning rank 8/16/32 (alpha scaled proportionally, alpha =
2 * r, a common LoRA heuristic), each trained briefly (`PROBE_MAX_STEPS`)
and compared by validation loss — a short probe, not a claim that these
numbers are the fully-converged final loss for each rank.

Everything else (dropout, target_modules) is held fixed at
`configs/default.yaml`'s values so this isolates rank as the variable
under test.


In [ ]:
import gc
import math
import time

from peft import LoraConfig, TaskType, get_peft_model
from transformers import Trainer, TrainingArguments

from lora_bench.training import CompletionOnlyDataCollator

collator = CompletionOnlyDataCollator(tokenizer=tokenizer, pad_to_multiple_of=8)

SWEEP_CANDIDATES = [
    {"r": 8, "lora_alpha": 16},
    {"r": 16, "lora_alpha": 32},
    {"r": 32, "lora_alpha": 64},
]
if SMOKE_TEST:
    SWEEP_CANDIDATES = SWEEP_CANDIDATES[:1]
PROBE_MAX_STEPS = 2 if SMOKE_TEST else 50
PROBE_BATCH_SIZE = 1 if SMOKE_TEST else 4
PROBE_GRAD_ACCUM = 1 if SMOKE_TEST else 4
SMOKE_ADAPTER_DIR = "/content/lora_bench_smoke_adapter"


def run_probe(candidate, tag):
    started_at = time.time()
    torch.cuda.reset_peak_memory_stats()
    model = load_base_model()
    lora_config = LoraConfig(
        r=candidate["r"],
        lora_alpha=candidate["lora_alpha"],
        lora_dropout=cfg.lora.dropout,
        target_modules=cfg.lora.target_modules,
        task_type=TaskType.CAUSAL_LM,
        bias="none",
    )
    model = get_peft_model(model, lora_config)

    args = TrainingArguments(
        output_dir=f"/content/sweep_{tag}",
        per_device_train_batch_size=PROBE_BATCH_SIZE,
        gradient_accumulation_steps=PROBE_GRAD_ACCUM,
        max_steps=PROBE_MAX_STEPS,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=max(1, int(0.03 * PROBE_MAX_STEPS)),
        logging_steps=1 if SMOKE_TEST else 10,
        save_strategy="no",
        eval_strategy="no",
        optim="paged_adamw_8bit",
        bf16=(COMPUTE_DTYPE == torch.bfloat16),
        fp16=(COMPUTE_DTYPE == torch.float16),
        report_to="none",
        seed=42,
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collator)
    train_result = trainer.train()
    eval_metrics = trainer.evaluate(eval_dataset=val_ds)
    assert math.isfinite(eval_metrics["eval_loss"]), "Non-finite validation loss"
    if SMOKE_TEST:
        model.save_pretrained(SMOKE_ADAPTER_DIR)
        tokenizer.save_pretrained(SMOKE_ADAPTER_DIR)

    result = {
        "val_loss": eval_metrics["eval_loss"],
        "train_metrics": train_result.metrics,
        "log_history": trainer.state.log_history,
        "elapsed_seconds": time.time() - started_at,
        "peak_vram_bytes": torch.cuda.max_memory_allocated(),
    }

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    return result

In [ ]:
sweep_results = []
for candidate in SWEEP_CANDIDATES:
    tag = f"r{candidate['r']}"
    print(f"--- probing {tag} (alpha={candidate['lora_alpha']}) ---")
    probe_result = run_probe(candidate, tag)
    sweep_results.append({**candidate, **probe_result})
    print(f"{tag}: val_loss={probe_result['val_loss']:.4f}")

print()
print(json.dumps(sweep_results, indent=2))
with open("/content/lora_sweep_results.json", "w") as f:
    json.dump(sweep_results, f, indent=2)

In [ ]:
assert (not SMOKE_TEST) or len(sweep_results) == 1
assert (not SMOKE_TEST) or sweep_results[0]["r"] == 8
winner = min(sweep_results, key=lambda r: r["val_loss"])
print(
    f"Winning config: r={winner['r']}, lora_alpha={winner['lora_alpha']}, "
    f"val_loss={winner['val_loss']:.4f}"
)
print()
if SMOKE_TEST:
    print("Smoke mode validates the stack only; this is not a rank-sweep decision.")
else:
    print(
        "Return lora_sweep_results.json for ADR-0006's defended rank decision. "
        "Never report these numbers before they come from a real run."
    )

## 5. Full fine-tune with the winning config

Same setup as each sweep probe, but a full multi-epoch run instead of a
50-step probe, using whichever config `winner` above resolved to.


In [ ]:
from peft import PeftModel

if SMOKE_TEST:
    # run_probe already trained and saved the single two-step rank-8 adapter.
    model = PeftModel.from_pretrained(load_base_model(), SMOKE_ADAPTER_DIR)
    model.eval()
    print("Reloaded the smoke adapter into a fresh base model.")
else:
    FULL_EPOCHS = 3
    FULL_BATCH_SIZE = 4
    FULL_GRAD_ACCUM = 4

    model = load_base_model()
    lora_config = LoraConfig(
        r=winner["r"],
        lora_alpha=winner["lora_alpha"],
        lora_dropout=cfg.lora.dropout,
        target_modules=cfg.lora.target_modules,
        task_type=TaskType.CAUSAL_LM,
        bias="none",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    steps_per_epoch = max(1, len(train_ds) // (FULL_BATCH_SIZE * FULL_GRAD_ACCUM))
    total_steps = steps_per_epoch * FULL_EPOCHS
    print(f"steps/epoch: {steps_per_epoch}  total steps: {total_steps}")

    full_args = TrainingArguments(
        output_dir="/content/lora_bench_finetune",
        per_device_train_batch_size=FULL_BATCH_SIZE,
        gradient_accumulation_steps=FULL_GRAD_ACCUM,
        num_train_epochs=FULL_EPOCHS,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=max(1, int(0.03 * total_steps)),
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        optim="paged_adamw_8bit",
        bf16=(COMPUTE_DTYPE == torch.bfloat16),
        fp16=(COMPUTE_DTYPE == torch.float16),
        report_to="none",
        seed=42,
    )

    trainer = Trainer(
        model=model,
        args=full_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
    )
    train_result = trainer.train()
    print(train_result)

In [ ]:
ADAPTER_DIR = SMOKE_ADAPTER_DIR if SMOKE_TEST else "/content/lora_bench_adapter"
if not SMOKE_TEST:
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter + tokenizer available at {ADAPTER_DIR}")

## 6. Quick qualitative check

Not Day 3's real benchmark harness (quality/latency/memory/cost) — just a
fast, human-readable sanity check that the fine-tuned model's outputs look
different from (and hopefully better than) the base model's, on a couple
of held-out validation examples. Real quality numbers come from Day 3.


In [ ]:
from lora_bench.data.cvefixes import to_chat_messages


def generate(gen_model, example, max_new_tokens=300):
    messages = to_chat_messages(example)[:1]  # user turn only, no ground-truth fix
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        output_ids = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)


if SMOKE_TEST:
    smoke_example = val_examples_all[0]
    smoke_output = generate(model, smoke_example, max_new_tokens=64)
    assert smoke_output.strip(), "Reloaded smoke adapter generated an empty response"
    RELOADED_GENERATION = {
        "cve_id": smoke_example.cve_id,
        "input": smoke_example.input,
        "ground_truth": smoke_example.output,
        "generated": smoke_output,
    }
    with open("/content/reloaded_generation.json", "w") as f:
        json.dump(RELOADED_GENERATION, f, indent=2)
    print(json.dumps(RELOADED_GENERATION, indent=2))
else:
    base_model_for_compare = load_base_model()
    for example in val_examples_all[:3]:
        print("=" * 80)
        print(f"CWE: {example.cwe_id} {example.cwe_name}")
        print("--- vulnerable code ---")
        print(example.input)
        print("--- ground-truth fix ---")
        print(example.output)
        print("--- base model output ---")
        print(generate(base_model_for_compare, example))
        print("--- fine-tuned model output ---")
        print(generate(model, example))

## Next

- In the default smoke mode, return `lora_bench_smoke_artifacts.zip`
  from the final cell for review. Do not set `SMOKE_TEST = False` until
  the 4-bit load/train/reload evidence in that ZIP passes review.
- After smoke validation, run normal mode and return its sweep results so
  ADR-0006 can defend the final LoRA rank with real numbers.
- Day 3 (not yet built): quantize the fine-tuned model to GGUF or AWQ, and
  add the repo-side (non-GPU, testable) benchmark harness cells to this
  same notebook -- quality, latency, memory, cost per 1K tokens, base vs.
  fine-tuned vs. quantized.


In [ ]:
import shutil
from pathlib import Path

from google.colab import files

RUN_METADATA.update(
    {
        "elapsed_seconds": time.time() - RUN_STARTED_AT,
        "preprocessing": PREPROCESSING_COUNTERS,
        "probe_results": sweep_results,
        "pip_freeze": subprocess.run(
            [sys.executable, "-m", "pip", "freeze"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.splitlines(),
    }
)
artifact_name = "lora_bench_smoke_artifacts" if SMOKE_TEST else "lora_bench_training_artifacts"
artifact_dir = Path("/content") / artifact_name
if artifact_dir.exists():
    shutil.rmtree(artifact_dir)
artifact_dir.mkdir()
shutil.copytree(ADAPTER_DIR, artifact_dir / "adapter")
for source in [
    "data/processed/manifest.json",
    "/content/preprocessing_counters.json",
    "/content/lora_sweep_results.json",
    "/content/reloaded_generation.json",
]:
    path = Path(source)
    if path.exists():
        shutil.copy2(path, artifact_dir / path.name)
with (artifact_dir / "run_metadata.json").open("w") as f:
    json.dump(RUN_METADATA, f, indent=2)
archive_path = shutil.make_archive(str(artifact_dir), "zip", artifact_dir)
print(f"Packaged smoke evidence: {archive_path}")
files.download(archive_path)